# Rerun 4 (GPU) — re-implemented SemEval-2025 systems on the same partitions (Paket D; R2-2)

Reviewer 2 asked for a feasible representative of recent emotion systems that use **adapters** (UoB-NLP, Laureano De Leon et al., SemEval-2025) and **class-balanced objectives** (Team UBD, SemEval-2025), re-implemented under the same partitions, label access and selection budget. This notebook trains both on the pooled SemEval-2025 Task 11 data (Sundanese, Javanese, Indonesian) of the **same ten seeds and splits as notebook 03**.

| System | What is re-implemented | Not reproduced (and why) |
|---|---|---|
| **UoB-NLP-style task adapter** | Pfeiffer-style bottleneck adapter after the feed-forward block of every layer (reduction factor 16), frozen encoder, adapter + classification head trained with binary cross-entropy; per-emotion thresholds on the validation partition | Language adapters (MLM-pretrained per language, 30k–100k steps) and their stacking: no unlabeled corpora/compute; UoB-NLP also report the task-only adapter as a variant |
| **Team UBD-style balanced fine-tuning** | Only the last two transformer blocks and the head are trained; emotion- and class-balanced BCE (per emotion: mean loss of positives and of negatives averaged; then averaged over emotions); balanced sampling (emotion and label drawn uniformly, then a matching text); AdamW lr 1e-5, wd 1e-2, effective batch 256, clipping 3, ≤ 500 steps, validation every 25 steps; per-emotion logit thresholds in [−2, 2] (step 0.1) | Their English/German/Spanish emotion-pretrained checkpoints and NLLB translation: no such checkpoints for Sundanese/Javanese/Indonesian, so the same multilingual encoders as in this study are used |

Both systems are run on **XLM-RoBERTa-base** (the encoder of UoB-NLP) and **multilingual-E5-base** (the encoder of this study). Label access matches the frozen pipeline: parameters are fitted on the training partition and every selection (early stopping, thresholds) uses only the validation partition; the test partition is used once. Results are compared with the frozen pipeline of notebook 03 by `scripts/analyze_paketD.py` (corrected resampled t-test over the same seeds).

**Kaggle**: *Add Input* SemEval-2025 Task 11 (optional: output of notebook 03 for a split check). Accelerator **GPU T4**, Internet **On**. Expected ≈ 4–6 h; resumable; stops cleanly before `TIME_BUDGET_H`.

In [ ]:
!pip -q install iterative-stratification
import os, sys, re, time, json, glob, random, warnings, gc, platform, shutil, math, copy, subprocess
from pathlib import Path
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.optim.lr_scheduler import LambdaLR
import sklearn, transformers
from sklearn.metrics import f1_score
from transformers import AutoTokenizer, AutoModel
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
warnings.filterwarnings("ignore"); transformers.logging.set_verbosity_error()
SESSION_T0 = time.time()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu"); USE_AMP = DEVICE.type == "cuda"
print("torch", torch.__version__, "| transformers", transformers.__version__, "| device", DEVICE)

## 1. Configuration

In [ ]:
SMOKE_TEST = bool(int(os.environ.get("MEXBAL_SMOKE", "0")))
SEEDS = [42, 7, 13, 77, 101, 202, 303, 404, 505, 606]
VAL_FRAC, TEST_FRAC = 0.15, 0.20
ENCODERS = {"xlmr": dict(name=os.environ.get("MEXBAL_FT_MODEL", "xlm-roberta-base"), prefix=None, pooling="first"),
            "e5":   dict(name=os.environ.get("MEXBAL_FT_MODEL", "intfloat/multilingual-e5-base"), prefix="query: ", pooling="mean")}
SYSTEMS = ["UoB-adapter", "UBD-balanced"]
MAX_LEN = 128
# UoB-NLP-style task adapter (task-adapter hyper-parameters are not reported by UoB-NLP -> AdapterHub defaults)
ADP = dict(reduction=16, lr=1e-4, wd=0.01, batch=16, max_epochs=30, patience=5, warmup=0.1)
# Team UBD-style balanced fine-tuning (as published)
UBD = dict(lr=1e-5, wd=1e-2, batch=256, micro_batch=64, clip=3.0, max_steps=500, eval_every=25, train_last_blocks=2)
TH_GRID_PROB = np.round(np.arange(0.05, 0.951, 0.01), 2)        # adapter: thresholds on probabilities
TH_GRID_LOGIT = np.round(np.arange(-2.0, 2.0001, 0.1), 1)        # UBD: thresholds on logits (as published)
TIME_BUDGET_H = 11.3
if SMOKE_TEST:
    SEEDS = [42, 7]; ADP.update(max_epochs=2, patience=1); UBD.update(batch=8, micro_batch=4, max_steps=6, eval_every=3)

INPUT_ROOTS = ("/kaggle/input", os.environ.get("MEXBAL_DATA", "/nonexistent"))
def _semeval_root():
    hits = [h for r in INPUT_ROOTS for h in sorted(glob.glob(os.path.join(r, "**", "track_a/train/sun.csv"), recursive=True))]
    hits.sort(key=lambda h: "public_data_test" not in h)
    for h in hits:
        root = Path(h).parents[2]
        need = [root / "track_a/dev/sun.csv", root / "track_c/dev/jav.csv", root / "track_c/dev/ind.csv"]
        if all(f.exists() for f in need) and all({"anger", "joy"} <= set(pd.read_csv(f, nrows=1).columns) for f in need):
            return str(root)
    return None
SEMEVAL_ROOT = _semeval_root(); assert SEMEVAL_ROOT, "SemEval data not found"
WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.environ.get("MEXBAL_WORK", "./work")
OUT_ROOT = os.path.join(WORK, "rerun_paketD")
_prev = [p for r in INPUT_ROOTS for p in glob.glob(os.path.join(r, "**", "rerun_paketD"), recursive=True) if os.path.isdir(p)]
if _prev and not os.path.exists(OUT_ROOT): shutil.copytree(_prev[0], OUT_ROOT); print("Resumed from", _prev[0])
for d in [OUT_ROOT, f"{OUT_ROOT}/predictions"]: Path(d).mkdir(parents=True, exist_ok=True)
RAW, HIST = f"{OUT_ROOT}/paketD_SemEval_raw.csv", f"{OUT_ROOT}/paketD_history.csv"

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)
def append_csv(path, rows):
    new = pd.DataFrame(rows); old = pd.read_csv(path) if os.path.exists(path) else pd.DataFrame()
    df = pd.concat([old, new], ignore_index=True); df.to_csv(path, index=False); return df
def done_keys(path, cols):
    if not os.path.exists(path): return set()
    return set(map(tuple, pd.read_csv(path)[cols].astype(str).values.tolist()))
hours_used = lambda: (time.time() - SESSION_T0) / 3600

## 2. Data and splits (identical to notebook 03)

In [ ]:
SEMEVAL_LABELS = ["anger", "disgust", "fear", "joy", "sadness", "surprise"]; LANGS = ["sun", "jav", "ind"]; K = 6
def load_semeval():
    sun = pd.concat([pd.read_csv(f"{SEMEVAL_ROOT}/track_a/train/sun.csv"), pd.read_csv(f"{SEMEVAL_ROOT}/track_a/dev/sun.csv")], ignore_index=True)
    data = {"sun": sun.reset_index(drop=True), "jav": pd.read_csv(f"{SEMEVAL_ROOT}/track_c/dev/jav.csv").reset_index(drop=True),
            "ind": pd.read_csv(f"{SEMEVAL_ROOT}/track_c/dev/ind.csv").reset_index(drop=True)}
    for lang, df in data.items(): df["uid"] = [f"{lang}_{i:05d}" for i in range(len(df))]
    return data
def _two_stage(splitter, y, seed):
    hold = VAL_FRAC + TEST_FRAC
    tr, h = next(splitter(n_splits=1, test_size=hold, random_state=seed).split(np.zeros(len(y)), y))
    va, te = next(splitter(n_splits=1, test_size=TEST_FRAC / hold, random_state=seed).split(np.zeros(len(h)), y[h]))
    return np.sort(tr), np.sort(h[va]), np.sort(h[te])
SEMEVAL_DATA = load_semeval()
SPLITS = {(l, s): _two_stage(MultilabelStratifiedShuffleSplit, SEMEVAL_DATA[l][SEMEVAL_LABELS].values.astype(int), s) for l in LANGS for s in SEEDS}
# global arrays: all texts of the three languages
TEXTS = np.concatenate([SEMEVAL_DATA[l]["text"].astype(str).values for l in LANGS])
Y = np.vstack([SEMEVAL_DATA[l][SEMEVAL_LABELS].values.astype(int) for l in LANGS])
UID = np.concatenate([SEMEVAL_DATA[l]["uid"].values for l in LANGS])
OFF = dict(zip(LANGS, np.cumsum([0] + [len(SEMEVAL_DATA[l]) for l in LANGS])[:-1]))
def parts(seed):
    g = lambda k: np.concatenate([SPLITS[(l, seed)][k] + OFF[l] for l in LANGS])
    return g(0), g(1), {l: SPLITS[(l, seed)][2] + OFF[l] for l in LANGS}
# optional check against notebook 03
_man = [h for r in INPUT_ROOTS for h in glob.glob(os.path.join(r, "**", "rerun_frozen", "split_manifest_all.csv"), recursive=True)]
if _man:
    m = pd.read_csv(_man[0]); m = m[m.dataset == "SemEval"]
    for s in SEEDS:
        tr, va, te = parts(s)
        for sp, idx in [("train", tr), ("val", va), ("test", np.concatenate(list(te.values())))]:
            assert set(m[(m.seed == s) & (m.split == sp)].uid) == set(UID[idx]), f"split differs: seed {s} {sp}"
    print("Splits identical to notebook 03")
tr, va, te = parts(42); print("seed 42 train/val/test:", len(tr), len(va), {l: len(v) for l, v in te.items()})

## 3. Models: bottleneck task adapter (UoB-NLP style) and partially unfrozen encoder (Team UBD style)

In [ ]:
class Adapter(nn.Module):
    # Pfeiffer-style bottleneck adapter: x + up(relu(down(x))), zero-initialised up-projection
    def __init__(self, d, reduction):
        super().__init__(); r = max(1, d // reduction)
        self.down, self.up = nn.Linear(d, r), nn.Linear(r, d); nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)
    def forward(self, x): return x + self.up(F.relu(self.down(x)))

class WithAdapter(nn.Module):
    def __init__(self, dense, reduction):
        super().__init__(); self.dense = dense; self.adapter = Adapter(dense.out_features, reduction)
    def forward(self, x): return self.adapter(self.dense(x))

def masked_mean(h, m):
    m = m.unsqueeze(-1).to(h.dtype); return (h * m).sum(1) / m.sum(1).clamp(min=1e-9)

BASE = {}
def base_encoder(enc):
    if enc not in BASE: BASE.clear(); BASE[enc] = AutoModel.from_pretrained(ENCODERS[enc]["name"])
    return BASE[enc]

class Net(nn.Module):
    def __init__(self, enc, system):
        super().__init__()
        self.enc = copy.deepcopy(base_encoder(enc)); self.pooling = ENCODERS[enc]["pooling"]
        for p in self.enc.parameters(): p.requires_grad = False
        layers = self.enc.encoder.layer
        if system == "UoB-adapter":
            for L in layers: L.output.dense = WithAdapter(L.output.dense, ADP["reduction"])   # after the FFN, before residual + LayerNorm
        else:
            for L in layers[-UBD["train_last_blocks"]:]:
                for p in L.parameters(): p.requires_grad = True
        H = self.enc.config.hidden_size; self.drop = nn.Dropout(0.1); self.head = nn.Linear(H, K)
    def forward(self, ids, mask):
        h = self.enc(input_ids=ids, attention_mask=mask).last_hidden_state
        z = h[:, 0] if self.pooling == "first" else masked_mean(h, mask)
        return self.head(self.drop(z))

TOK, ENC_IDS = {}, {}
def tokenise(enc):
    if enc not in ENC_IDS:
        tok = AutoTokenizer.from_pretrained(ENCODERS[enc]["name"]); TOK[enc] = tok
        ENC_IDS[enc] = tok([(ENCODERS[enc]["prefix"] or "") + t for t in TEXTS], truncation=True, max_length=MAX_LEN)["input_ids"]
    return TOK[enc], ENC_IDS[enc]

def to_tensors(enc, b):
    tok, ids_all = tokenise(enc); seqs = [ids_all[i] for i in b]; L = max(map(len, seqs))
    ids = torch.full((len(seqs), L), tok.pad_token_id, dtype=torch.long); mask = torch.zeros((len(seqs), L), dtype=torch.long)
    for k, s in enumerate(seqs): ids[k, :len(s)] = torch.tensor(s); mask[k, :len(s)] = 1
    return ids.to(DEVICE), mask.to(DEVICE)
def autocast(): return torch.autocast(device_type="cuda", dtype=torch.float16, enabled=USE_AMP) if USE_AMP else torch.autocast("cpu", enabled=False)

@torch.no_grad()
def logits_of(model, enc, idx, bs=64):
    model.eval(); idx = np.asarray(idx); _, ids_all = tokenise(enc)
    order = np.argsort([len(ids_all[i]) for i in idx], kind="stable"); out = []
    for k in range(0, len(idx), bs):
        ids, mask = to_tensors(enc, idx[order][k:k + bs])
        with autocast(): out.append(model(ids, mask).float().cpu().numpy())
    Z = np.vstack(out); res = np.empty_like(Z); res[order] = Z; return res

def tune_thresholds(S, y, grid):
    th = []
    for j in range(K):
        sc = [f1_score(y[:, j], (S[:, j] >= t).astype(int), zero_division=0) for t in grid]
        th.append(float(grid[int(np.argmax(sc))]))
    return np.asarray(th)
macro = lambda y, yp: f1_score(y, yp, average="macro", zero_division=0)
trainable_state = lambda m: {k: v.detach().cpu().clone() for k, v in m.state_dict().items() if k in {n for n, p in m.named_parameters() if p.requires_grad}}

## 4. Training procedures

In [ ]:
def train_adapter(enc, tr, va, seed):
    set_seed(seed); model = Net(enc, "UoB-adapter").to(DEVICE)
    params = [p for p in model.parameters() if p.requires_grad]
    steps = math.ceil(len(tr) / ADP["batch"]); total = steps * ADP["max_epochs"]; warm = max(1, int(ADP["warmup"] * total))
    opt = torch.optim.AdamW(params, lr=ADP["lr"], weight_decay=ADP["wd"])
    sched = LambdaLR(opt, lambda s: (s + 1) / warm if s < warm else max(0.0, (total - s) / max(1, total - warm)))
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP); rng = np.random.default_rng(seed)
    best, best_ep, state, bad, hist, t0 = -1.0, 0, None, 0, [], time.perf_counter()
    for ep in range(1, ADP["max_epochs"] + 1):
        model.train(); perm = np.asarray(tr)[rng.permutation(len(tr))]; tl = 0.0
        for k in range(0, len(perm), ADP["batch"]):
            b = perm[k:k + ADP["batch"]]; ids, mask = to_tensors(enc, b); yt = torch.tensor(Y[b], dtype=torch.float, device=DEVICE)
            with autocast(): z = model(ids, mask)
            loss = F.binary_cross_entropy_with_logits(z.float(), yt)
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(params, 1.0); scaler.step(opt); scaler.update(); sched.step(); tl += loss.item()
        P = 1 / (1 + np.exp(-logits_of(model, enc, va))); f = macro(Y[va], (P >= 0.5).astype(int))
        hist.append({"epoch": ep, "train_loss": tl / steps, "val_macro_f1@0.5": f})
        if f > best + 1e-6: best, best_ep, state, bad = f, ep, trainable_state(model), 0
        else:
            bad += 1
            if bad >= ADP["patience"]: break
    model.load_state_dict(state, strict=False)
    Pv = 1 / (1 + np.exp(-logits_of(model, enc, va))); th = tune_thresholds(Pv, Y[va], TH_GRID_PROB)
    score = lambda idx: 1 / (1 + np.exp(-logits_of(model, enc, idx)))
    return model, score, th, dict(best_val=best, best_epoch=best_ep, epochs_run=len(hist), train_time_s=time.perf_counter() - t0), hist

def balanced_batch(tr, n, rng):
    # Team UBD sampling: draw an emotion and a label uniformly, then a training text with that label for that emotion
    out = []
    for _ in range(n):
        e = rng.integers(K); lab = rng.integers(2); pool = tr[Y[tr, e] == lab]
        if len(pool) == 0: pool = tr
        out.append(rng.choice(pool))
    return np.asarray(out)

def balanced_bce(z, y):
    # per emotion: mean BCE of positives and mean BCE of negatives, averaged; then averaged over emotions
    l = F.binary_cross_entropy_with_logits(z, y, reduction="none"); per = []
    for j in range(y.shape[1]):
        pos, neg = y[:, j] > 0.5, y[:, j] < 0.5; terms = []
        if pos.any(): terms.append(l[pos, j].mean())
        if neg.any(): terms.append(l[neg, j].mean())
        per.append(torch.stack(terms).mean())
    return torch.stack(per).mean()

def train_ubd(enc, tr, va, seed):
    set_seed(seed); model = Net(enc, "UBD-balanced").to(DEVICE); rng = np.random.default_rng(seed)
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=UBD["lr"], weight_decay=UBD["wd"]); scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    acc = UBD["batch"] // UBD["micro_batch"]; best, best_step, state, hist, t0 = -1.0, 0, None, [], time.perf_counter()
    for step in range(1, UBD["max_steps"] + 1):
        model.train(); opt.zero_grad(set_to_none=True); tl = 0.0
        for _ in range(acc):
            b = balanced_batch(np.asarray(tr), UBD["micro_batch"], rng); ids, mask = to_tensors(enc, b)
            yt = torch.tensor(Y[b], dtype=torch.float, device=DEVICE)
            with autocast(): z = model(ids, mask)
            loss = balanced_bce(z.float(), yt) / acc; scaler.scale(loss).backward(); tl += loss.item()
        scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(params, UBD["clip"]); scaler.step(opt); scaler.update()
        if step % UBD["eval_every"] == 0 or step == UBD["max_steps"]:
            Zv = logits_of(model, enc, va); th = tune_thresholds(Zv, Y[va], TH_GRID_LOGIT)
            f = macro(Y[va], (Zv >= th[None, :]).astype(int)); hist.append({"step": step, "train_loss": tl, "val_macro_f1_tuned": f})
            if f > best + 1e-6: best, best_step, state = f, step, trainable_state(model)
    model.load_state_dict(state, strict=False)
    Zv = logits_of(model, enc, va); th = tune_thresholds(Zv, Y[va], TH_GRID_LOGIT)
    return model, (lambda idx: logits_of(model, enc, idx)), th, dict(best_val=best, best_step=best_step, train_time_s=time.perf_counter() - t0), hist

## 5. Runs (10 seeds × 2 systems × 2 encoders; resumable)

In [ ]:
stop = False
for seed in SEEDS:
    if stop: break
    tr, va, te = parts(seed); done = done_keys(RAW, ["system", "encoder", "seed"])
    for system in SYSTEMS:
        for enc in ENCODERS:
            if (system, enc, str(seed)) in done: continue
            if hours_used() > TIME_BUDGET_H: print("Time budget reached; re-run with this output attached."); stop = True; break
            fit = train_adapter if system == "UoB-adapter" else train_ubd
            model, score, th, info, hist = fit(enc, tr, va, seed)
            rows, YT, YP, art = [], [], [], {"thresholds": th, **info}
            for l in LANGS:
                S = score(te[l]); yp = (S >= th[None, :]).astype(int); yt = Y[te[l]]
                rows.append({"system": system, "encoder": enc, "seed": seed, "lang": l, "n_test": len(yt), "macro_f1": macro(yt, yp)})
                YT.append(yt); YP.append(yp); art[l] = {"test_uid": UID[te[l]], "y_true": yt, "y_pred": yp, "score": S.astype(np.float32)}
            YT, YP = np.vstack(YT), np.vstack(YP)
            rows.append({"system": system, "encoder": enc, "seed": seed, "lang": "pooled", "n_test": len(YT), "macro_f1": macro(YT, YP)})
            rows.append({"system": system, "encoder": enc, "seed": seed, "lang": "average", "n_test": len(YT), "macro_f1": float(np.mean([r["macro_f1"] for r in rows[:3]]))})
            for r in rows: r.update({k: v for k, v in info.items()}, thresholds=json.dumps([float(t) for t in th]))
            append_csv(RAW, rows); append_csv(HIST, [{"run": f"{system}|{enc}|{seed}", **h} for h in hist])
            import joblib; joblib.dump(art, f"{OUT_ROOT}/predictions/{system}_{enc}_seed{seed}.joblib", compress=3)
            print(f"{system:13s} {enc:5s} seed={seed:<3} avg={rows[-1]['macro_f1']:.4f} pooled={rows[-2]['macro_f1']:.4f} "
                  f"({info['train_time_s']:.0f}s) | {hours_used():.2f} h")
            del model; gc.collect(); torch.cuda.empty_cache() if torch.cuda.is_available() else None
        if stop: break

## 6. Summary and archive

In [ ]:
if os.path.exists(RAW):
    r = pd.read_csv(RAW)
    print(r.pivot_table(index=["system", "encoder"], columns="lang", values="macro_f1", aggfunc="mean")[["sun", "jav", "ind", "average", "pooled"]].round(4))
    print(r.groupby(["system", "encoder"]).seed.nunique())
env = {"python": platform.python_version(), "torch": torch.__version__, "transformers": transformers.__version__, "sklearn": sklearn.__version__,
       "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None, "encoders": ENCODERS, "adapter": ADP, "ubd": UBD,
       "seeds": SEEDS, "smoke_test": SMOKE_TEST, "session_hours": round(hours_used(), 2)}
json.dump(env, open(f"{OUT_ROOT}/environment_paketD.json", "w"), indent=2)
shutil.make_archive(f"{WORK}/rerun_paketD_outputs", "zip", OUT_ROOT); print("Archive:", f"{WORK}/rerun_paketD_outputs.zip")